# Advanced Batch Prediction Example

This notebook demonstrates:
1. Creating predictions with idempotency keys
2. Handling pagination for large batches
3. Robust error handling and retries
4. Downloading and processing GeoTIFF results

## Setup

In [ ]:
import os
import sys
import time
import uuid
import httpx
from pathlib import Path
from spatialise import SpatialiseSoilPrediction
import spatialise

# Initialize the client with custom timeout
client = SpatialiseSoilPrediction(
    api_key=os.environ.get("SPATIALISE_API_KEY"),
    timeout=60.0,  # 60 seconds timeout
)

print("✓ Client initialized with 60s timeout")

## Helper Functions

Define reusable functions for batch creation, polling, and downloading.

In [ ]:
def create_batch_with_retry(jobs, metadata=None, max_retries=3):
    """Create a batch with retry logic and idempotency."""
    # Generate a unique idempotency key to prevent duplicate submissions
    idempotency_key = str(uuid.uuid4())

    for attempt in range(max_retries):
        try:
            batch = client.batch.create(
                jobs=jobs,
                metadata=metadata,
                idempotency_key=idempotency_key,
            )
            return batch
        except spatialise.APIConnectionError as e:
            print(f"⚠ Connection error (attempt {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                time.sleep(2**attempt)  # Exponential backoff
            else:
                raise
        except spatialise.RateLimitError as e:
            print(f"⚠ Rate limit exceeded: {e}")
            print("  Please wait before retrying.")
            raise
        except spatialise.AuthenticationError as e:
            print(f"✗ Authentication failed: {e}")
            print("  Please check your API key.")
            raise
        except spatialise.APIStatusError as e:
            print(f"✗ API error (status {e.status_code}): {e}")
            raise


def poll_batch_status(batch_id, timeout=300, poll_interval=5):
    """Poll batch status until completion or timeout.

    Args:
        batch_id: The batch ID to poll
        timeout: Maximum time to wait in seconds (default: 5 minutes)
        poll_interval: Time between polls in seconds (default: 5 seconds)

    Returns:
        BatchRetrieveStatusResponse when batch is complete
    """
    start_time = time.time()

    while time.time() - start_time < timeout:
        try:
            status = client.batch.retrieve_status(batch_id)

            elapsed = int(time.time() - start_time)
            print(
                f"  [{elapsed}s] {status.status}: "
                f"{status.completed_jobs}/{status.total_jobs} completed, "
                f"{status.failed_jobs} failed, "
                f"{status.pending_jobs} pending"
            )

            if status.status in ["completed", "failed"]:
                return status

            time.sleep(poll_interval)

        except spatialise.APIConnectionError as e:
            print(f"  ⚠ Connection error while polling: {e}")
            time.sleep(poll_interval * 2)  # Wait longer on connection errors
        except spatialise.NotFoundError:
            print(f"  ✗ Batch {batch_id} not found")
            raise

    raise TimeoutError(f"Batch {batch_id} did not complete within {timeout} seconds")


def get_all_jobs(batch_id):
    """Retrieve all jobs from a batch, handling pagination.

    Args:
        batch_id: The batch ID to retrieve jobs from

    Yields:
        Individual job objects
    """
    cursor = None

    while True:
        status = client.batch.retrieve_status(
            batch_id,
            cursor=cursor,
            limit=100,  # Maximum allowed per page
        )

        for job in status.jobs:
            yield job

        if not status.has_more:
            break

        cursor = status.next_cursor


def download_geotiff(url, output_path):
    """Download a GeoTIFF from a signed URL.

    Args:
        url: The signed URL to download from
        output_path: Path to save the GeoTIFF file
    """
    print(f"  Downloading to {output_path}...")
    try:
        with httpx.stream("GET", url, timeout=60.0) as response:
            response.raise_for_status()

            with open(output_path, "wb") as f:
                for chunk in response.iter_bytes(chunk_size=8192):
                    f.write(chunk)

        print(f"  ✓ Downloaded {output_path.stat().st_size:,} bytes")
    except httpx.HTTPError as e:
        print(f"  ✗ Download failed: {e}")
        raise


print("✓ Helper functions defined")

## Step 1: Create Multi-Year Batch

Create a batch with multiple years of predictions for the same area.

In [ ]:
# Define multiple prediction jobs for different years
jobs = [
    {
        "year": year,
        "polygon": {
            "type": "Polygon",
            "coordinates": [
                [
                    [6.7, 52.8],
                    [6.70074, 52.8],
                    [6.70074, 52.80045],
                    [6.7, 52.80045],
                    [6.7, 52.8],
                ]
            ],
        },
    }
    for year in [2020, 2021, 2022, 2023]
]

print(f"Creating batch with {len(jobs)} jobs...")

try:
    batch = create_batch_with_retry(
        jobs=jobs,
        metadata={
            "project": "Multi-year soil analysis",
            "location": "Netherlands",
            "description": "Soil organic carbon trends 2020-2023",
        },
    )
    print(f"\n✓ Batch created: {batch.batch_id}")
    print(f"  Total jobs: {batch.total_jobs}")
    print(f"  Status: {batch.status}")
except Exception as e:
    print(f"✗ Failed to create batch: {e}")
    raise

## Step 2: Poll for Completion

Wait for all predictions to complete with timeout protection.

In [ ]:
print("\nWaiting for predictions to complete...\n")

try:
    final_status = poll_batch_status(batch.batch_id, timeout=600)
    print(f"\n✓ Batch {final_status.status}!")
except TimeoutError as e:
    print(f"✗ {e}")
    raise
except Exception as e:
    print(f"✗ Error polling batch: {e}")
    raise

## Step 3: Process Results with Pagination

Retrieve all jobs and download the GeoTIFF files.

In [ ]:
print("\nProcessing results...\n")

output_dir = Path("results")
output_dir.mkdir(exist_ok=True)

completed_count = 0
failed_count = 0

for job in get_all_jobs(batch.batch_id):
    print(f"Job {job.job_id}:")

    if job.status == "completed" and job.signed_cog_url:
        # Download the GeoTIFF
        output_file = output_dir / f"prediction_{job.job_id}.tif"
        try:
            download_geotiff(job.signed_cog_url, output_file)
            completed_count += 1
        except Exception as e:
            print(f"  ⚠ Failed to download: {e}")
            failed_count += 1

    elif job.status == "failed":
        print(f"  ✗ Job failed: {job.error_message}")
        failed_count += 1

    else:
        print(f"  Status: {job.status}")

    print()

# Summary
print("=" * 60)
print(f"Summary:")
print(f"  Total jobs: {final_status.total_jobs}")
print(f"  Successfully downloaded: {completed_count}")
print(f"  Failed: {failed_count}")
print(f"  Results saved to: {output_dir.absolute()}")
print("=" * 60)

## Step 4: Analyze Downloaded Files

List all downloaded GeoTIFF files and their sizes.

In [ ]:
print("\nDownloaded files:")
print("=" * 60)

total_size = 0
for file in sorted(output_dir.glob("*.tif")):
    size = file.stat().st_size
    total_size += size
    print(f"  {file.name}: {size:,} bytes ({size / 1024 / 1024:.2f} MB)")

print("=" * 60)
print(f"Total: {total_size:,} bytes ({total_size / 1024 / 1024:.2f} MB)")

## Optional: Re-check Batch Status Later

You can retrieve the batch status at any time using the batch ID.

In [ ]:
# Re-check status (e.g., if signed URLs expired and need to be refreshed)
refreshed_status = client.batch.retrieve_status(batch.batch_id)

print(f"Batch {batch.batch_id}:")
print(f"  Status: {refreshed_status.status}")
print(f"  Completed: {refreshed_status.completed_jobs}/{refreshed_status.total_jobs}")
print(f"  Failed: {refreshed_status.failed_jobs}")
print(f"  Last updated: {refreshed_status.updated_at}")